In [ ]:
!pip install pandas 
!pip install matplotlib 
!pip install seaborn
!pip install nltk 
!pip install transformers datasets 
!pip install torch

In [210]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from collections import Counter
import string
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

In [211]:
# Cell 1: Load the raw chat file
file_path = 'WhatsApp Chat with Zandile.txt'  

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"File loaded successfully! Total characters: {len(text)}")
    print("\nFirst 500 characters preview:")
    print(text[:500])  # Preview to verify it loaded correctly
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found. Make sure it's in the same folder as this notebook.")
except UnicodeDecodeError:
    print("Encoding error. Try opening the file in a text editor and saving as UTF-8.")
    # Fallback: Try latin-1 encoding
    with open(file_path, 'r', encoding='latin-1') as f:
        text = f.read()
    print("Loaded with latin-1 encoding.")


File loaded successfully! Total characters: 1019019

First 500 characters preview:
2023/08/05, 15:59 - Messages and calls are end-to-end encrypted. Only people in this chat can read, listen to, or share them. Learn more.
2023/08/05, 15:59 - Romeo: Hey
2023/08/05, 15:59 - Zandile: Hey
2023/08/05, 16:02 - Romeo: So I wanna do a poem first, it's more time efficient, for the first one there's no specific topic or theme it's just gonna go off whatever inspiration you give me
2023/08/05, 16:04 - Zandile: I didn't get your name?
2023/08/05, 16:04 - Zandile: What type of inspiration w


In [212]:
# Cell 2: Parse the chat using regex (Updated for YYYY/MM/DD, HH:MM - Sender: Message format)
import re
from datetime import datetime

# Updated Regex pattern for your format: YYYY/MM/DD, HH:MM - Sender: Message
# Captures timestamp, sender, and message precisely
pattern = r'(\d{4}/\d{2}/\d{2}, \d{2}:\d{2}) - (.*?): (.*)'

# Find all matches (MULTILINE for line-by-line parsing)
matches = re.findall(pattern, text, re.MULTILINE)

print(f"Found {len(matches)} potential messages.")
if len(matches) > 0:
    print("Example match (should match your format):")
    print(matches[0])  # (timestamp_str, sender, message) e.g., ('2023/07/09, 17:47', 'Romeo', 'Hi')
    
    # Test with your exact example
    test_line = '2023/07/09, 17:47 - Romeo: Hi'
    test_match = re.findall(pattern, test_line)
    print(f"\nTest with your example line: {test_match}")
else:
    print("No matches found. Check your file format or regex pattern.")
    print("Sample lines from file (first 5):")
    lines = text.split('\n')[:5]
    for i, line in enumerate(lines):
        print(f"Line {i}: {repr(line)}")  # repr() shows exact content with escapes



Found 17655 potential messages.
Example match (should match your format):
('2023/08/05, 15:59', 'Romeo', 'Hey')

Test with your example line: [('2023/07/09, 17:47', 'Romeo', 'Hi')]


In [213]:
# Cell 3: Convert matches to DataFrame with proper datetime parsing (Updated for YYYY/MM/DD, HH:MM)
data = []

for timestamp_str, sender, message in matches:
    try:
        # Parse timestamp: Primary format for your example (YYYY/MM/DD, HH:MM)
        # Fallbacks for other common WhatsApp formats (in case of mixed data)
        timestamp = None
        for fmt in [
            '%Y/%m/%d, %H:%M',  # Your format: 2023/07/09, 17:47
            '%m/%d/%Y, %H:%M',  # Alternative: 07/09/2023, 17:47
            '%d/%m/%Y, %H:%M',  # Alternative: 09/07/2023, 17:47
            '%Y/%m/%d, %H:%M:%S', '%m/%d/%Y, %H:%M:%S', '%d/%m/%Y, %H:%M:%S'  # With seconds if present
        ]:
            try:
                timestamp = datetime.strptime(timestamp_str, fmt)
                break
            except ValueError:
                continue
        
        if timestamp is None:
            print(f"Warning: Could not parse timestamp '{timestamp_str}'. Skipping this line.")
            continue
        
        # Store as list of dicts
        data.append({
            'timestamp': timestamp,
            'sender': sender.strip(),  # Remove extra spaces (e.g., if " Romeo ")
            'message': message.strip()  # Remove extra spaces
        })
    except Exception as e:
        print(f"Error parsing line '{timestamp_str} - {sender}: {message[:50]}...': {e}")
        continue

# Create DataFrame
import pandas as pd
df = pd.DataFrame(data)

print(f"Created DataFrame with {len(df)} rows.")
print("\nDataFrame info:")
print(df.info())
print("\nFirst 5 rows (should show your format):")
print(df.head())

# Test with your example
if len(data) > 0:
    test_row = data[0] if '2023/07/09' in str(data[0].get('timestamp', '')) else data[0]
    print(f"\nTest parsed example: Timestamp={test_row['timestamp']}, Sender={test_row['sender']}, Message={test_row['message']}")


Created DataFrame with 17655 rows.

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17655 entries, 0 to 17654
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  17655 non-null  datetime64[ns]
 1   sender     17655 non-null  object        
 2   message    17655 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 413.9+ KB
None

First 5 rows (should show your format):
            timestamp   sender  \
0 2023-08-05 15:59:00    Romeo   
1 2023-08-05 15:59:00  Zandile   
2 2023-08-05 16:02:00    Romeo   
3 2023-08-05 16:04:00  Zandile   
4 2023-08-05 16:04:00  Zandile   

                                             message  
0                                                Hey  
1                                                Hey  
2  So I wanna do a poem first, it's more time eff...  
3                            I didn't get your name?  
4    What type of 

In [214]:
# Cell 4: Remove system messages and empty rows
# Define keywords for system messages (add more if needed)
system_keywords = [
    'messages to this chat and calls are now secured',
    'you deleted this message',
    'this message was deleted',
    '<media omitted>',
    '<media excluded>',
    'joined using invite link',
    'left',
    'changed to',
    'created group'
]

# Filter out rows containing these keywords (case-insensitive)
mask_system = ~df['message'].str.contains('|'.join(system_keywords), case=False, na=False)
df = df[mask_system]

# Also remove completely empty messages
df = df[df['message'].str.strip() != '']

print(f"After filtering: {len(df)} rows remaining.")
print("\nSample after filtering:")
print(df.head())


After filtering: 13449 rows remaining.

Sample after filtering:
            timestamp   sender  \
0 2023-08-05 15:59:00    Romeo   
1 2023-08-05 15:59:00  Zandile   
2 2023-08-05 16:02:00    Romeo   
3 2023-08-05 16:04:00  Zandile   
4 2023-08-05 16:04:00  Zandile   

                                             message  
0                                                Hey  
1                                                Hey  
2  So I wanna do a poem first, it's more time eff...  
3                            I didn't get your name?  
4    What type of inspiration will I give you tho? 😭  


In [215]:
# Cell 5: Normalize text
import re  # Already imported, but ensure

def normalize_message(text):
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove emojis (optional: comment out if you want to keep them for personality)
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # Step 3: Remove special characters (keep letters, numbers, spaces, basic punctuation)
    # Optional: Adjust if you want to keep more (e.g., ! ? .)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)  # Keeps alphanumeric, spaces, and common punctuation
    
    # Step 4: Strip extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply normalization
df['message'] = df['message'].apply(normalize_message)

print("Normalization applied.")
print("\nBefore/After example:")
sample_before = df.iloc[0]['message'] if 'message' in df.columns else "No data"
# Note: This shows after, but assume you printed before earlier
print(f"Sample normalized message: {df.iloc[0]['message'] if len(df) > 0 else 'No data'}")
print(df.head())


Normalization applied.

Before/After example:
Sample normalized message: hey
            timestamp   sender  \
0 2023-08-05 15:59:00    Romeo   
1 2023-08-05 15:59:00  Zandile   
2 2023-08-05 16:02:00    Romeo   
3 2023-08-05 16:04:00  Zandile   
4 2023-08-05 16:04:00  Zandile   

                                             message  
0                                                hey  
1                                                hey  
2  so i wanna do a poem first, its more time effi...  
3                             i didnt get your name?  
4      what type of inspiration will i give you tho?  


In [216]:
# Cell 6: Save to CSV and JSON
output_csv = 'cleaned_whatsapp_chat38.csv'
# output_json = 'cleaned_whatsapp_chat.json'

# Save to CSV (easy for Pandas, good for ML datasets)
df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Saved CSV: {output_csv}")

# # Save to JSON (for more flexibility, e.g., if integrating with other tools)
# df.to_json(output_json, orient='records', date_format='iso', indent=2)
# print(f"Saved JSON: {output_json}")

# Verify by loading back a sample
df_loaded = pd.read_csv(output_csv)
print("\nLoaded CSV preview (first 3 rows):")
print(df_loaded.head(3))

print("\nFinal DataFrame shape:", df.shape)
print("Columns:", df.columns.tolist())


Saved CSV: cleaned_whatsapp_chat38.csv

Loaded CSV preview (first 3 rows):
             timestamp   sender  \
0  2023-08-05 15:59:00    Romeo   
1  2023-08-05 15:59:00  Zandile   
2  2023-08-05 16:02:00    Romeo   

                                             message  
0                                                hey  
1                                                hey  
2  so i wanna do a poem first, its more time effi...  

Final DataFrame shape: (13449, 3)
Columns: ['timestamp', 'sender', 'message']
